# 🎭 Avatar 3D Pipeline — Full Training Notebook

Notebook này chứa toàn bộ 6 giai đoạn training cho Avatar 3D Pipeline.

**Cách sử dụng:**
- Chạy lần lượt từng Section (Phase) theo thứ tự.
- Nếu bị ngắt kết nối, chạy lại từ **Phase 0** (Setup) rồi nhảy đến Phase bạn đang dở.
- Tất cả checkpoint đều lưu trên Google Drive, AI sẽ tự resume.

**Yêu cầu:** GPU T4 trở lên. Vào `Runtime → Change runtime type → GPU`.

In [ ]:
# === Fix: Tạo symlinks đúng cách trên /content/ (local disk) ===
import os, subprocess

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
WEIGHTS_DIR = f"{DRIVE_BASE}/avatar_weights"
DATA_DIR    = f"{DRIVE_BASE}/avatar_data"
CKPT_DIR    = f"{DRIVE_BASE}/avatar_checkpoints"

# Tạo thư mục trên Drive nếu chưa có
for d in [
    f"{WEIGHTS_DIR}/deca", f"{WEIGHTS_DIR}/flame2020",
    f"{DATA_DIR}/selfies", f"{DATA_DIR}/albedo",
    f"{DATA_DIR}/normal", f"{DATA_DIR}/vietnamese/selfies",
    f"{CKPT_DIR}/geometry", f"{CKPT_DIR}/texture", f"{CKPT_DIR}/texture_lora",
]:
    os.makedirs(d, exist_ok=True)

# Tạo symlinks tại /content/ (RAM disk, hỗ trợ symlink)
for src, dst in [(WEIGHTS_DIR, "/content/weights"), (DATA_DIR, "/content/data")]:
    if os.path.lexists(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f"✅ {dst} → {src}")

# Cài deps (bỏ qua nếu đã cài)
os.chdir("/content/drive/MyDrive/avatar_project/avatar_3d_pipeline")
print("✅ Working dir:", os.getcwd())


---
## ⚙️ Phase 0: Setup & Xác Minh Môi Trường
**Chạy mỗi khi mở notebook hoặc bị mất kết nối.**

In [2]:
# === Cell 0.1: Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
# 1. Thuấn di (Dịch chuyển tức thời) ra khỏi ổ đĩa bị sập để về lại bộ nhớ gốc của Google
os.chdir('/content')

# 2. Xóa xổ cái dây cáp Drive đang bị đứt nửa chừng
!umount -f /content/drive || true
!rm -rf /content/drive

# 3. Ép cắm cáp Google Drive lại từ đầu 100% sạch sẽ
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
# Force remount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [3]:
# === Cell 0.2: Đi vào thư mục code & Cài đặt môi trường ===
# Thay đổi đường dẫn nếu bạn đặt code ở vị trí khác
%cd /content/drive/MyDrive/avatar_project/avatar_3d_pipeline
!bash scripts/setup_colab.sh

/content/drive/MyDrive/avatar_project/avatar_3d_pipeline
🚀 Avatar 3D Pipeline — Colab Setup
📌 PyTorch: 2.10.0+cu128 | CUDA: 12.8

📦 Installing core dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 6.7 MB/s eta 0:00:00

📦 Installing PyTorch Geometric...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.9 MB/s eta 0:00:0000:01
⚠️  Some PyG extensions failed — GNN fallback mode will be used

📦 Installing PyTorch3D...
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done

📦 Setting up DECA (decalib)...
  ✅ DECA repo cloned
     ━━━━━━━━━━━━

In [ ]:
# === Cell 0.3: Kiểm tra Weights đã có đủ chưa ===
from pathlib import Path

print("🔍 Kiểm tra file trọng số (Weights):")
print("=" * 50)

# Symlinks giờ nằm tại /content/weights (không phải ./weights)
weight_files = {
    "/content/weights/deca/deca_model.tar": "DECA Model (bắt buộc)",
    "/content/weights/flame2020/generic_model.pkl": "FLAME 2020 Model (bắt buộc)",
    "/content/weights/flame2020/FLAME_texture.npz": "FLAME Texture Space (bắt buộc)",
    "/content/weights/flame2020/head_template.obj": "Head Template Mesh (bắt buộc)",
}

all_ok = True
for wf, desc in weight_files.items():
    p = Path(wf)
    if p.exists():
        size_mb = p.stat().st_size / 1024 / 1024
        print(f"  ✅ {desc}")
        print(f"     └── {wf} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {desc}")
        print(f"     └── THIẾU: {wf}")
        all_ok = False

print("=" * 50)
if all_ok:
    print("🎉 Tất cả weights sẵn sàng!")
else:
    print("⛔ Thiếu file! Upload lên avatar_project/avatar_weights/ trước.")


In [3]:
# === Cell 0.4: Kiểm tra GPU ===
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU: {gpu_name}")
    print(f"   VRAM: {vram_gb:.1f} GB")
    print(f"   PyTorch: {torch.__version__}")
    print(f"   CUDA: {torch.version.cuda}")
else:
    print("⛔ KHÔNG CÓ GPU!")
    print("   Vào Runtime → Change runtime type → Chọn GPU")

✅ GPU: NVIDIA L4
   VRAM: 22.0 GB
   PyTorch: 2.10.0+cu128
   CUDA: 12.8


---
## 📸 Phase 1: Tải Ảnh FFHQ & Sinh Dữ Liệu Training
**Mục đích:** Tải 5000 ảnh mặt người đa dạng, rồi tạo ra bộ (selfie, albedo, normal) để dạy AI.

In [ ]:
# === Cell 1.1 (FAST + FIXED): Tải ảnh khuôn mặt song song 8 luồng ===
!pip install -q datasets

import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
output_dir = Path(f"{DRIVE_BASE}/avatar_data/ffhq_raw")
output_dir.mkdir(parents=True, exist_ok=True)

NUM_SAMPLES = 5000
NUM_WORKERS = 8

# Đếm ảnh đã có (cả PNG lẫn JPG)
existing_stems = set(p.stem for p in output_dir.glob("*.*"))
print(f"📂 Đã có sẵn: {len(existing_stems)} ảnh")

if len(existing_stems) >= NUM_SAMPLES - 10:
    print("✅ Đủ rồi. Bỏ qua tải.")
else:
    remaining = NUM_SAMPLES - len(existing_stems)
    print(f"📥 Cần tải thêm: {remaining} ảnh (8 luồng song song)...")

    # Bỏ qua FFHQ (cần auth), dùng thẳng huggan/celeba
    from datasets import load_dataset 
    ds = load_dataset("student/celebA", split=f"train[:{NUM_SAMPLES}]")

    def save_one(args):
        i, sample = args
        stem = f"{i:05d}"
        if stem in existing_stems:
            return i  # Bỏ qua ảnh đã tải
        path = output_dir / f"{stem}.png"
        img = sample["image"]
        if img.mode != "RGB":
            img = img.convert("RGB")
        img.save(str(path), format="PNG")
        return i

    done = 0
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = {pool.submit(save_one, (i, s)): i for i, s in enumerate(ds)}
        for future in as_completed(futures):
            done += 1
            if done % 500 == 0 or done == NUM_SAMPLES:
                total_now = len(list(output_dir.glob("*.png")))
                print(f"  📸 {done}/{NUM_SAMPLES} xử lý | {total_now} file trên Drive...")

    total = len(list(output_dir.glob("*.png")))
    print(f"\n✅ Hoàn tất! Tổng: {total} ảnh trong {output_dir}")


In [ ]:
# === Cell 1.1b: Kiểm tra tiến độ đếm số ảnh ===
import os
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
output_dir = Path(f"{DRIVE_BASE}/avatar_data/ffhq_raw")

if output_dir.exists():
    count_png = len(list(output_dir.glob("*.png")))
    count_jpg = len(list(output_dir.glob("*.jpg")))
    total = count_png + count_jpg
    
    print("="*40)
    print(f"📊 TIẾN ĐỘ TẢI DỮ LIỆU:")
    print(f"   📁 Thư mục: {output_dir}")
    print(f"   🖼️ Số ảnh .PNG : {count_png}")
    print(f"   🖼️ Số ảnh .JPG : {count_jpg}")
    print("-" * 40)
    print(f"   ✅ TỔNG CỘNG  : {total} / 5000 ảnh")
    print("="*40)
else:
    print("⛔ Thư mục chưa được tạo. Hãy chạy Cell tải trước.")


In [ ]:
# === VÁ extract_tex + xóa ảnh cũ fallback ===

# 1. Vá reconstructor.py trên Drive
path = '/content/drive/MyDrive/avatar_project/avatar_3d_pipeline/core/reconstructor.py'
with open(path, 'r') as f:
    code = f.read()

old = """            if hasattr(deca_cfg, "model") and hasattr(deca_cfg.model, "use_tex"):
                deca_cfg.model.use_tex = True
            if hasattr(deca_cfg, "model") and hasattr(deca_cfg.model, "extract_tex"):
                deca_cfg.model.extract_tex = True"""

new = """            deca_cfg.model.defrost()
            deca_cfg.model.use_tex = True
            deca_cfg.model.extract_tex = True
            deca_cfg.model.freeze()"""

if old in code:
    code = code.replace(old, new)
    with open(path, 'w') as f:
        f.write(code)
    print("✅ reconstructor.py — extract_tex FIXED!")
else:
    print("⚠️ Không tìm thấy đoạn cũ, thử vá trực tiếp...")
    # Fallback: tìm hasattr pattern
    code = code.replace(
        'if hasattr(deca_cfg, "model") and hasattr(deca_cfg.model, "use_tex"):\n                deca_cfg.model.use_tex = True',
        'deca_cfg.model.defrost()\n            deca_cfg.model.use_tex = True\n            deca_cfg.model.extract_tex = True\n            deca_cfg.model.freeze()'
    )
    with open(path, 'w') as f:
        f.write(code)
    print("✅ Vá fallback xong!")

# 2. Xóa ảnh fallback cũ (từ ảnh 2141 trở đi đều là fallback)
import glob, os
albedo_files = sorted(glob.glob('/content/data/albedo/*.jpg'))
normal_files = sorted(glob.glob('/content/data/normal/*.jpg'))
selfie_files = sorted(glob.glob('/content/data/selfie/*.jpg'))

# Xóa TẤT CẢ để chạy lại từ đầu với DECA thật
count = 0
for f in albedo_files + normal_files + selfie_files:
    os.remove(f)
    count += 1
print(f"🗑️ Đã xóa {count} file cũ (toàn fallback)")
print("🔄 Chạy lại Cell 1.2 để sinh ảnh DECA thật!")


In [ ]:
# === Cell Dọn Rác (Chạy 1 lần) ===
import os
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
output_dir = Path(f"{DRIVE_BASE}/avatar_data/ffhq_raw")

# Tìm các file PNG bị trùng với JPG
png_files = list(output_dir.glob("*.png"))
deleted = 0

for p in png_files:
    # Nếu như tấm PNG này đã có anh em song sinh ở bản JPG -> Dứt khoát tiễn PNG đi
    if (output_dir / f"{p.stem}.jpg").exists():
        os.remove(p)
        deleted += 1

print(f"🧹 Đã quét và vứt bỏ {deleted} ảnh đuôi PNG bị trùng lặp!")


In [ ]:
# === Cell 1.2: Sinh cặp dữ liệu (Selfie → Albedo + Normal) ===
# ⏱️ ~2.5 giờ cho 5000 ảnh (DECA thật) | Có resume nếu bị dừng

%cd /content/drive/MyDrive/avatar_project/avatar_3d_pipeline

# --- Bước 1: Patch Python 3.12 cho subprocess ---
# (vì !python tạo process mới, không thừa hưởng patches trong kernel)

# Patch chumpy nếu chưa
import site, os
for sp in site.getsitepackages():
    ch_init = os.path.join(sp, 'chumpy', '__init__.py')
    if os.path.exists(ch_init):
        with open(ch_init, 'r') as f:
            c = f.read()
        if "from numpy import bool" in c:
            c = c.replace(
                "from numpy import bool, int, float, complex, object, unicode, str, nan, inf",
                "from numpy import nan, inf"
            )
            with open(ch_init, 'w') as f:
                f.write(c)
            print("✅ chumpy patched")
        
    ch_py = os.path.join(sp, 'chumpy', 'ch.py')
    if os.path.exists(ch_py):
        with open(ch_py, 'r') as f:
            c2 = f.read()
        if "getargspec" in c2 and "getfullargspec" not in c2:
            c2 = c2.replace("inspect.getargspec", "inspect.getfullargspec")
            with open(ch_py, 'w') as f:
                f.write(c2)
            print("✅ chumpy/ch.py patched")

# Copy FLAME files nếu chưa có
import shutil
flame_src = '/content/drive/MyDrive/avatar_weights/flame2020'
deca_data = '/content/DECA/data'
if os.path.isdir(flame_src):
    for f in os.listdir(flame_src):
        dst = os.path.join(deca_data, f)
        if not os.path.exists(dst):
            shutil.copy2(os.path.join(flame_src, f), dst)
            print(f"✅ Copied {f}")

# --- Bước 2: Chạy DECA ---
!python training/generate_synthetic_pairs.py \
    --input-dir /content/data/ffhq_raw \
    --output-dir /content/data \
    --ext .jpg \
    --image-size 512 \
    --max-samples 5000 \
    --weights-dir /content/weights \
    --resume


In [ ]:
# === COPY DỮ LIỆU TỪ Ổ CỨNG VÀO DRIVE ===
import shutil, os

src_base = '/content/data'
dst_base = '/content/drive/MyDrive/avatar_project/generated_data'

# Thư mục thực tế tên là 'selfies' (có chữ s)
for folder in ['selfies', 'albedo', 'normal']:
    src_dir = os.path.join(src_base, folder)
    dst_dir = os.path.join(dst_base, folder)
    
    if os.path.exists(src_dir):
        os.makedirs(dst_dir, exist_ok=True)
        files = os.listdir(src_dir)
        print(f"📦 Đang copy {len(files)} file từ {folder} sang Drive...")
        
        # Copy từng file
        for f in files:
            shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
            
        print(f"  ✅ Đã copy xong {folder}!")
    else:
        print(f"❌ Không tìm thấy foder gốc: {src_dir}")

print(f"\n🎉 XONG! Hãy vào Google Drive của bạn kiểm tra đường dẫn: Avatar_project/generated_data !")


In [ ]:
# === Cell 1.3: Kiểm tra chất lượng dữ liệu vừa sinh ===
%cd /content/drive/MyDrive/avatar_project/avatar_3d_pipeline
DRIVE_BASE = "/content/drive/MyDrive/avatar_project"

!python training/validate_dataset.py \
    --data-root {DRIVE_BASE}/avatar_data \
    --min-size 256 --fix


In [ ]:
# === Cell 1.4: Xem thử mẫu dữ liệu đã sinh ===
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import random

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
data_root = Path(f"{DRIVE_BASE}/avatar_data")

selfie_files = sorted((data_root / "selfies").glob("*.png"))
if selfie_files:
    samples = random.sample(selfie_files, min(3, len(selfie_files)))
    fig, axes = plt.subplots(len(samples), 3, figsize=(12, 4 * len(samples)))
    if len(samples) == 1:
        axes = [axes]

    for i, sf in enumerate(samples):
        stem = sf.stem
        selfie = Image.open(sf)
        albedo = Image.open(data_root / "albedo" / f"{stem}.png")
        normal = Image.open(data_root / "normal" / f"{stem}.png")

        axes[i][0].imshow(selfie); axes[i][0].set_title("Selfie")
        axes[i][1].imshow(albedo); axes[i][1].set_title("Albedo")
        axes[i][2].imshow(normal); axes[i][2].set_title("Normal")
        for ax in axes[i]:
            ax.axis("off")

    plt.suptitle("🎨 Mẫu dữ liệu: Selfie → Albedo → Normal", fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f"📊 Tổng số mẫu: {len(selfie_files)}")
else:
    print("❌ Chưa có dữ liệu. Chạy Cell 1.2 trước.")

In [ ]:
# === HIỂN THỊ HEAT MAP SO SÁNH: DECA 3D vs FALLBACK 2D ===
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

# Cấu hình đường dẫn
data_dir = '/content/drive/MyDrive/avatar_project/avatar_data'
selfies_dir = os.path.join(data_dir, 'selfies')
albedo_dir  = os.path.join(data_dir, 'albedo')
normal_dir  = os.path.join(data_dir, 'normal')

files = os.listdir(selfies_dir)
if not files:
    print("❌ Chưa có file nào trong thư mục selfies!")
else:
    test_file = files[0] # Hoặc đổi số [0] thành số khác để test ảnh khác
    
    # --- 1. LOAD & TẠO DATA ---
    selfie_img = cv2.imread(os.path.join(selfies_dir, test_file))
    selfie_img = cv2.cvtColor(selfie_img, cv2.COLOR_BGR2RGB)
    
    # DECA
    deca_albedo = cv2.cvtColor(cv2.imread(os.path.join(albedo_dir, test_file)), cv2.COLOR_BGR2RGB)
    deca_normal = cv2.cvtColor(cv2.imread(os.path.join(normal_dir, test_file)), cv2.COLOR_BGR2RGB)
    
    # Fallback (Bản cũ hay bị dùng khi lỗi)
    fall_albedo = cv2.bilateralFilter(selfie_img, d=9, sigmaColor=75, sigmaSpace=75)
    gray = cv2.cvtColor(selfie_img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    dx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=5)
    dy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=5)
    dx = dx / (np.max(np.abs(dx)) + 1e-8) * 2.0
    dy = dy / (np.max(np.abs(dy)) + 1e-8) * 2.0
    fall_normal = np.zeros((*gray.shape, 3), dtype=np.float32)
    fall_normal[:, :, 0] = -dx        # R: X
    fall_normal[:, :, 1] = -dy        # G: Y
    fall_normal[:, :, 2] = 1.0        # B: Z
    norm = np.sqrt(np.sum(fall_normal ** 2, axis=-1, keepdims=True))
    fall_normal = (((fall_normal / (norm + 1e-8)) + 1.0) * 0.5 * 255.0).clip(0, 255).astype(np.uint8)

    # --- 2. TÍNH TOÁN MAP ---
    # Heatmap Albedo (Độ sai lệch pixel giữa cấu trúc gốc và cấu trúc mới lột)
    # DECA lột da 3D nên heatmap ra hình rất độc đáo. Fallback thì tối thui vì không lột.
    diff_deca = np.mean(np.abs(selfie_img.astype(float) - deca_albedo.astype(float)), axis=-1)
    diff_fall = np.mean(np.abs(selfie_img.astype(float) - fall_albedo.astype(float)), axis=-1)
    
    # Heatmap độ sâu trục Z (Z-Depth Geometry) từ Normal Map
    # Channel B của ảnh màu RGB lưu trữ vector Z hướng ra ngoài (chiều sâu mũi, mắt...)
    z_deca = deca_normal[:, :, 2]
    z_fall = fall_normal[:, :, 2]
    
    # --- 3. VẼ BIỂU ĐỒ ---
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.subplots_adjust(hspace=0.3)
    
    # HÀNG 1: DECA 3D ENGINE
    axes[0,0].imshow(deca_albedo)
    axes[0,0].set_title(f"DECA (CHÍNH THỨC)\nBóc Tách Texture UV")
    axes[0,0].axis('off')
    
    im1 = axes[0,1].imshow(diff_deca, cmap='magma', vmin=0, vmax=255) # Magma cho cấu trúc bóc tách
    axes[0,1].set_title("Heat Map (Cấu Trúc Lột Dãn)\nÁnh sáng mạnh = Lột mảng 3D rộng")
    fig.colorbar(im1, ax=axes[0,1], shrink=0.7)
    axes[0,1].axis('off')
    
    im2 = axes[0,2].imshow(z_deca, cmap='viridis', vmin=100, vmax=255) # Viridis cho độ sâu
    axes[0,2].set_title("Geometry Z-Depth Heat Map\nMũi nhô cao, hốc mắt sâu")
    fig.colorbar(im2, ax=axes[0,2], shrink=0.7)
    axes[0,2].axis('off')

    # HÀNG 2: FALLBACK 2D THUẬT TOÁN CŨ
    axes[1,0].imshow(fall_albedo)
    axes[1,0].set_title(f"FALLBACK (BỘ LỌC CŨ)\nGiữ nguyên ảnh 2D")
    axes[1,0].axis('off')
    
    im3 = axes[1,1].imshow(diff_fall, cmap='magma', vmin=0, vmax=255)
    axes[1,1].set_title("Heat Map (Cấu Trúc Lột Dãn)\nTối thui = Không bóc tách được da mặt")
    fig.colorbar(im3, ax=axes[1,1], shrink=0.7)
    axes[1,1].axis('off')
    
    im4 = axes[1,2].imshow(z_fall, cmap='viridis', vmin=100, vmax=255)
    axes[1,2].set_title("Geometry Z-Depth Heat Map\nPhẳng lỳ như tờ giấy")
    fig.colorbar(im4, ax=axes[1,2], shrink=0.7)
    axes[1,2].axis('off')

    plt.suptitle("🔬 PHÂN TÍCH CHUYÊN SÂU: SỰ VƯỢT TRỘI CỦA PIPELINE DECA 3D", fontsize=16, fontweight='bold', y=0.98)
    plt.show()


---
## 🦴 Phase 2: Train Geometry VAE (Nặn Xương 3D)
**Mục đích:** Dạy AI cách biến các điểm lưới 3D (mesh vertices) thành khuôn mặt mịn màng, chuẩn hình thể.

**Thời gian ước tính:** ~1-2 giờ (100 epochs trên T4 GPU)

In [1]:
# === Di chuyển .npy từ meshes/meshes/ ra meshes/ ===
import shutil
from pathlib import Path

src = Path('/content/drive/MyDrive/avatar_project/avatar_data/meshes/meshes')
dst = Path('/content/drive/MyDrive/avatar_project/avatar_data/meshes')

if not src.exists():
    print("❌ Không tìm thấy thư mục nguồn:", src)
else:
    files = list(src.glob('*.npy'))
    print(f"📦 Tìm thấy {len(files)} file .npy, bắt đầu di chuyển...")
    
    moved = 0
    skipped = 0
    for f in files:
        target = dst / f.name
        if target.exists():
            skipped += 1
        else:
            shutil.move(str(f), str(target))
            moved += 1
    
    print(f"✅ Đã di chuyển: {moved} file")
    print(f"⚡ Bỏ qua (đã tồn tại): {skipped} file")
    
    # Xóa thư mục con rỗng sau khi di chuyển xong
    remaining = list(src.glob('*'))
    if not remaining:
        src.rmdir()
        print("🗑️ Đã xóa thư mục con meshes/meshes/ (rỗng)")
    else:
        print(f"⚠️ Còn {len(remaining)} file trong meshes/meshes/, chưa xóa thư mục")


📦 Tìm thấy 5000 file .npy, bắt đầu di chuyển...
✅ Đã di chuyển: 5000 file
⚡ Bỏ qua (đã tồn tại): 0 file
🗑️ Đã xóa thư mục con meshes/meshes/ (rỗng)


In [ ]:
# === CELL KIỂM TRA CHẤT LƯỢNG MESH 3D (.npy) ===
import os
import numpy as np
import plotly.graph_objects as go

# Đường dẫn chuẩn xác
meshes_dir = '/content/drive/MyDrive/avatar_project/avatar_data/meshes/meshes'
faces_path = '/content/drive/MyDrive/avatar_project/avatar_data/meshes/faces.npy'


# Chỉ lấy các file lưới của ảnh, loại trừ file cấu trúc tam giác faces.npy
files = [f for f in os.listdir(meshes_dir) if f.endswith('.npy') and f != 'faces.npy']

if not files:
    print("❌ Chưa có file .npy nào được tạo ra.")
else:
    # Lấy đại 1 file đầu tiên để test
    test_file = files[0]
    vertices = np.load(os.path.join(meshes_dir, test_file))
    
    faces = None
    if os.path.exists(faces_path):
        faces = np.load(faces_path)

    print("=" * 50)
    print(f"✅ Đang phân tích file: {test_file}")
    print(f"📏 Hình dáng mảng dữ liệu (Shape): {vertices.shape}")
    
    # KHI DECA CHẠY CHUẨN, shape PHẢI LÀ (5023, 3). 
    # Tức là 5023 điểm, mỗi điểm có 3 toạ độ x, y, z.
    if vertices.shape == (5023, 3):
        print("🎯 ĐẠT CHUẨN: Đây chính xác là Topology chuẩn của mô hình mặt người FLAME!")
    else:
        print("⚠️ CẢNH BÁO: Hình dáng không đúng chuẩn FLAME (có thể là Fallback).")
    
    if faces is not None:
        print(f"📐 Số lượng đa giác (Faces): {faces.shape[0]} tam giác")
    print("=" * 50)
    
    print("🎨 Đang vẽ mô hình 3D... (Có thể mất 3-5 giây để hiển thị, dùng chuột để xoay)")
    
    # Vẽ mô hình 3D
    if faces is not None:
        fig = go.Figure(data=[
            go.Mesh3d(
                x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
                i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                color='rgb(255, 200, 180)', # Màu da người
                opacity=0.9,
                lighting=dict(ambient=0.4, diffuse=0.8, roughness=0.1, specular=0.5),
                lightposition=dict(x=100, y=100, z=100)
            )
        ])
    else:
        # Nếu chưa có lưới tam giác thì vẽ bằng các dấu chấm (Point Cloud)
        fig = go.Figure(data=[
            go.Scatter3d(
                x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
                mode='markers', marker=dict(size=2, color='blue')
            )
        ])
        
    fig.update_layout(
        title=f"Mô hình 3D: {test_file}",
        scene=dict(aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()


In [ ]:
# === CELL CHỮA CHÁY: COPY FILE BỊ THIẾU ===
import shutil, os

# Đường dẫn đúng
flame_src = '/content/drive/MyDrive/avatar_project/avatar_weights/flame2020'
deca_data = '/content/DECA/data'

os.makedirs(deca_data, exist_ok=True)

if os.path.isdir(flame_src):
    count = 0
    for f in os.listdir(flame_src):
        dst = os.path.join(deca_data, f)
        if not os.path.exists(dst):
            shutil.copy2(os.path.join(flame_src, f), dst)
            count += 1
            print(f"✅ Đã copy: {f}")
        else:
            print(f"⚡ Đã có sẵn: {f}")
    if count > 0:
        print("\n🎉 XONG! File mô hình DECA đã được nạp đủ. Bạn có thể chạy lại Cell 2.1 ngay bây giờ!")
else:
    print(f"❌ Không tìm thấy thư mục: {flame_src}")


✅ Đã copy: FLAME_texture.npz
✅ Đã copy: generic_model.pkl
⚡ Đã có sẵn: head_template.obj
✅ Đã copy: FLAME_albedo_from_BFM.npz

🎉 XONG! File mô hình DECA đã được nạp đủ. Bạn có thể chạy lại Cell 2.1 ngay bây giờ!


In [14]:
# === Cell 2.1: Sinh Mesh Data từ ảnh (DECA → .npy) ===
# ⏱️ Thời gian: ~40-60 phút cho 5000 ảnh | Bị ngắt → chạy lại tự resume ✅
%cd /content/drive/MyDrive/avatar_project/avatar_3d_pipeline
DRIVE_BASE = "/content/drive/MyDrive/avatar_project"

!python training/generate_mesh_data.py \
    --input-dir {DRIVE_BASE}/avatar_data/selfies \
    --output-dir {DRIVE_BASE}/avatar_data/meshes \
    --weights-dir /content/weights \
    --max-samples 5000 \
    --resume


/content/drive/MyDrive/avatar_project/avatar_3d_pipeline
📸 Found 5000 images in /content/drive/MyDrive/avatar_project/avatar_data/selfies
✅ DECA reconstructor loaded
⏩ Resume mode: skipping 295 already-processed meshes
Generating meshes:   0% 0/5000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
creating the FLAME Decoder
trained model found. load /content/weights/deca/deca_model.tar
/usr/local/lib/python3.12/dist-packages/pytorch3d/io/obj_io.py:550: UserWarning: Mtl file does not exist: /content/DECA/data/template.m

In [4]:
!pip install -q ipywidgets
from google.colab import output
output.enable_custom_widget_manager()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.4 MB/s eta 0:00:00


In [5]:
# === Cell 2.2: Train Geometry VAE (Final Optimized) ===
import sys, os, argparse, importlib

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
PIPELINE_PATH = f"{DRIVE_BASE}/avatar_3d_pipeline"

if PIPELINE_PATH not in sys.path:
    sys.path.insert(0, PIPELINE_PATH)
os.chdir(PIPELINE_PATH)

# Ép load lại code mới nhất từ file
import training.train_geometry_vae as _tgv
importlib.reload(_tgv)
GeometryVAETrainer = _tgv.GeometryVAETrainer

args = argparse.Namespace(
    data_root    = f"{DRIVE_BASE}/avatar_data/meshes",
    output_dir   = f"{DRIVE_BASE}/avatar_checkpoints/geometry",
    epochs       = 100,
    batch_size   = 32,
    lr           = 3e-4,
    weight_decay = 1e-5,
    max_grad_norm= 1.0,
    latent_dim   = 256,
    num_workers  = 0,
    max_samples  = None,
    save_every   = 5,
    compile      = False,
    cache_in_ram = True,
    w_recon      = 1.0,
    w_kl         = 0.001,
    w_laplacian  = 0.1,
    w_edge       = 0.05,
)

trainer = GeometryVAETrainer(args)
trainer.train()


📦 Caching 5000 meshes into RAM (Parallel I/O)...


Loading:   0%|          | 0/5000 [00:00<?, ?mesh/s]

✅ Cached 5000 meshes (575 MB used in RAM)
   Edge index cached: torch.Size([2, 29990])
   ✅ Encoder partitions cached (8 parts)
   ✅ Encoder batch partitions cached (B=32)
   ⚡ Batched graph tensors cached (B=32, N=5023)
🔄 Resuming from /content/drive/MyDrive/avatar_project/avatar_checkpoints/geometry/geometry_vae_latest.pt
🔧 Geometry VAE Trainer initialized
   Device: cuda
   Batch size: 32
   Samples: 5000
   Epochs: 100 (start from 95)
   Latent dim: 256

🚀 Bắt đầu training: Epoch 96 → 100
   Mỗi epoch = 157 steps | Log mỗi 15 steps

── Epoch 96/100 ──────────────────
  [  9.6%] 15/157 | loss=0.00038 | 0.7 step/s | ETA 218s
  [ 19.1%] 30/157 | loss=0.00038 | 0.7 step/s | ETA 192s
  [ 28.7%] 45/157 | loss=0.00038 | 0.7 step/s | ETA 168s
  [ 38.2%] 60/157 | loss=0.00038 | 0.7 step/s | ETA 145s
  [ 47.8%] 75/157 | loss=0.00038 | 0.7 step/s | ETA 123s
  [ 57.3%] 90/157 | loss=0.00038 | 0.7 step/s | ETA 100s
  [ 66.9%] 105/157 | loss=0.00038 | 0.7 step/s | ETA 78s
  [ 76.4%] 120/157 | lo

In [6]:
# === Cell 2.3: Kiểm tra kết quả Geometry Training ===
import json
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
ckpt_dir = Path(f"{DRIVE_BASE}/avatar_checkpoints/geometry")

# Tìm checkpoint mới nhất
latest = ckpt_dir / "geometry_vae_latest.pt"
if latest.exists():
    size_mb = latest.stat().st_size / 1024 / 1024
    print(f"✅ Checkpoint: {latest.name} ({size_mb:.1f} MB)")
else:
    print("❌ Chưa có checkpoint. Chạy Cell 2.2 trước.")

# Đọc metrics nếu có
metrics_path = ckpt_dir / "metrics_latest.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print(f"\n📊 Training Progress:")
    print(f"   Epoch: {metrics.get('epoch', '?')}")
    for k, v in metrics.items():
        if k != 'epoch':
            print(f"   {k}: {v:.6f}" if isinstance(v, float) else f"   {k}: {v}")
else:
    print("ℹ️ Chưa có metrics file.")

✅ Checkpoint: geometry_vae_latest.pt (34.3 MB)

📊 Training Progress:
   Epoch: 100
   total: 0.000376
   recon: 0.000361
   kl: 0.000203
   laplacian: 0.000003
   edge: 0.000304


---
## 🎨 Phase 3: Train Texture Networks (Vẽ Da)
**Mục đích:** Dạy AI vẽ da 3D siêu thực với 4 mạng chuyên gia:
- **Stage 1 (G):** Sinh texture cơ bản
- **Stage 2 (GA):** Chỉnh tông da
- **Stage 3 (GC):** Xóa bóng râm (De-lighting)
- **Stage 4 (GE):** Trích xuất Normal/Specular maps

**⚠️ Quan trọng:** Phải chạy theo đúng thứ tự Stage 1 → 2 → 3 → 4.

**Thời gian ước tính:** ~2-4 giờ mỗi Stage trên T4 GPU

In [ ]:
# === Cell 3.1: Stage 1 — Train G (Texture Generator) [A100 Optimized] ===
# ⏱️ Thời gian dự kiến: ~2.5-3 giờ (A100 40GB)
# 🔄 Auto-resume: Có
# 🚀 Tối ưu: torch.compile(max-autotune) + batch=32 + 8 workers

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
PIPELINE_DIR = f"{DRIVE_BASE}/avatar_3d_pipeline"

%cd {PIPELINE_DIR}
%env PYTORCH_ALLOC_CONF=expandable_segments:True

!python training/train_texture_full.py \
    --data-root {DRIVE_BASE}/avatar_data \
    --weights-dir {DRIVE_BASE}/avatar_weights \
    --output-dir {DRIVE_BASE}/avatar_checkpoints/texture \
    --stage 1 \
    --epochs 50 \
    --batch-size 32 \
    --image-size 512 \
    --num-workers 8 \
    --save-every 1


/content/drive/MyDrive/avatar_project/avatar_3d_pipeline
env: PYTORCH_ALLOC_CONF=expandable_segments:True
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 528M/528M [00:02<00:00, 245MB/s] 
🎨 Texture Trainer — Stage: 1
   Device: cuda
   Samples: 5000
   Epochs: 50 (start from 0)

🎨 Bắt đầu training Texture: Epoch 1 → 50
   Mỗi epoch = 625 steps | Log mỗi 62 steps

── Epoch 1/50 ──────────────────
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/functional.py:282: RuntimeWarning: invalid value encountered in cast
  npimg = (npimg * 255).astype(np.uint8)
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/functional.py:282: RuntimeWarning: invalid value encountered in cast
  npimg = (npimg * 255).astype(np.uint8)
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/functional.py:282: RuntimeWarning: invalid value encountered in cast
  npimg = (npimg * 255).astype(np.uint8

In [ ]:
# === Cell 3.2: Stage 2 — Train GA (Skin Tone Adjuster) ===
# ⚠️ Chạy SAU KHI Stage 1 hoàn tất!
# ⏱️ Thời gian: ~1-2 giờ

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"

!python training/train_texture_full.py \
    --data-root {DRIVE_BASE}/avatar_data \
    --weights-dir {DRIVE_BASE}/avatar_checkpoints/texture \
    --output-dir {DRIVE_BASE}/avatar_checkpoints/texture \
    --stage 2 \
    --epochs 30 \
    --batch-size 2 \
    --save-every 1

In [ ]:
# === Cell 3.3: Stage 3 — Train GC (De-lighting / Reflectance) ===
# ⚠️ Chạy SAU KHI Stage 2 hoàn tất!
# ⏱️ Thời gian: ~1-2 giờ

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"

!python training/train_texture_full.py \
    --data-root {DRIVE_BASE}/avatar_data \
    --weights-dir {DRIVE_BASE}/avatar_checkpoints/texture \
    --output-dir {DRIVE_BASE}/avatar_checkpoints/texture \
    --stage 3 \
    --epochs 30 \
    --batch-size 2 \
    --save-every 1

In [ ]:
# === Cell 3.4: Stage 4 — Train GE (PBR Extractor: Normal + Specular) ===
# ⚠️ Chạy SAU KHI Stage 3 hoàn tất!
# ⏱️ Thời gian: ~1 giờ

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"

!python training/train_texture_full.py \
    --data-root {DRIVE_BASE}/avatar_data \
    --weights-dir {DRIVE_BASE}/avatar_checkpoints/texture \
    --output-dir {DRIVE_BASE}/avatar_checkpoints/texture \
    --stage 4 \
    --epochs 20 \
    --batch-size 2 \
    --save-every 1

In [ ]:
# === Cell 3.5: Kiểm tra Texture Checkpoints ===
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
tex_dir = Path(f"{DRIVE_BASE}/avatar_checkpoints/texture")

expected_files = [
    "texture_g_latest.pt",
    "texture_ga_latest.pt",
    "texture_gc_latest.pt",
    "texture_ge_latest.pt",
]

print("📊 Trạng thái Texture Checkpoints:")
print("=" * 50)
for fname in expected_files:
    p = tex_dir / fname
    if p.exists():
        size_mb = p.stat().st_size / 1024 / 1024
        print(f"  ✅ {fname} ({size_mb:.1f} MB)")
    else:
        print(f"  ⬜ {fname} (chưa train)")
print("=" * 50)

---
## 🇻🇳 Phase 4: LoRA Fine-Tune Cho Khuôn Mặt Người Việt
**Mục đích:** Tinh chỉnh AI để avatar khớp với màu da và đặc trưng khuôn mặt châu Á / Việt Nam.

**Yêu cầu:** Đã upload 100-500 ảnh selfie người Việt vào:
`avatar_project/avatar_data/vietnamese/selfies/`

**Thời gian ước tính:** ~30-60 phút

In [ ]:
# === Cell 4.1: Kiểm tra ảnh người Việt đã upload ===
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
vn_dir = Path(f"{DRIVE_BASE}/avatar_data/vietnamese/selfies")

if vn_dir.exists():
    extensions = ["*.jpg", "*.jpeg", "*.png", "*.webp", "*.jfif"]
    files = []
    for ext in extensions:
        files.extend(vn_dir.glob(ext))
        files.extend(vn_dir.glob(ext.upper()))
    files = list(set(files))
    print(f"📸 Tìm thấy {len(files)} ảnh người Việt")
    if len(files) < 50:
        print("⚠️ Khuyến nghị tối thiểu 100 ảnh để fine-tune hiệu quả.")
    elif len(files) >= 100:
        print("✅ Đủ ảnh! Sẵn sàng fine-tune.")
else:
    print(f"❌ Thư mục chưa tồn tại: {vn_dir}")
    print("   Hãy tạo thư mục và upload ảnh selfie người Việt vào.")

In [ ]:
# === Cell 4.2: Sinh cặp dữ liệu cho ảnh VN ===
# ⏱️ Thời gian: ~5-15 phút

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"

!python training/generate_synthetic_pairs.py \
    --input-dir {DRIVE_BASE}/avatar_data/vietnamese/selfies \
    --output-dir {DRIVE_BASE}/avatar_data/vietnamese \
    --weights-dir weights \
    --image-size 512

In [ ]:
# === Cell 4.3: Fine-tune bằng LoRA ===
# ⏱️ Thời gian: ~30-60 phút
# 🔄 Auto-resume: Có

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"

!python training/fine_tune_texture.py \
    --data-root {DRIVE_BASE}/avatar_data/vietnamese \
    --weights-dir {DRIVE_BASE}/avatar_checkpoints/texture \
    --output-dir {DRIVE_BASE}/avatar_checkpoints/texture_lora \
    --batch-size 1 \
    --grad-accum-steps 8 \
    --lora-rank 8 \
    --lora-alpha 16 \
    --epochs 25

---
## 📊 Phase 5: Đánh Giá Chất Lượng (Evaluate)
**Mục đích:** Đo lường chất lượng avatar bằng các chỉ số khoa học (SSIM, LPIPS).

In [ ]:
# === Cell 5.1: Chạy đánh giá ===
DRIVE_BASE = "/content/drive/MyDrive/avatar_project"

!python training/evaluate.py \
    --pred-dir {DRIVE_BASE}/avatar_data/albedo \
    --gt-dir {DRIVE_BASE}/avatar_data/albedo \
    --metrics all \
    --image-size 512 \
    --report-path {DRIVE_BASE}/avatar_checkpoints/eval_report.json

In [ ]:
# === Cell 5.2: Xem báo cáo đánh giá ===
import json
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
report_path = Path(f"{DRIVE_BASE}/avatar_checkpoints/eval_report.json")

if report_path.exists():
    report = json.loads(report_path.read_text())
    print("📊 Kết Quả Đánh Giá:")
    print("=" * 50)
    for k, v in report.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
    print("=" * 50)
    print("\n💡 Mục tiêu:")
    print("   SSIM > 0.85 → Tốt")
    print("   LPIPS < 0.15 → Tốt (càng thấp càng giống thật)")
else:
    print("❌ Chưa có báo cáo. Chạy Cell 5.1 trước.")

---
## 📦 Phase 6: Xuất Weights Cho Production
**Mục đích:** Gom tất cả file `.pt` đã train xong, đóng gói sẵn sàng deploy lên Backend API.

In [ ]:
# === Cell 6.1: Tổng kết toàn bộ Weights đã Train ===
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"

production_files = {
    "Geometry VAE": f"{DRIVE_BASE}/avatar_checkpoints/geometry/geometry_vae_latest.pt",
    "Texture G": f"{DRIVE_BASE}/avatar_checkpoints/texture/texture_g_latest.pt",
    "Texture GA": f"{DRIVE_BASE}/avatar_checkpoints/texture/texture_ga_latest.pt",
    "Texture GC": f"{DRIVE_BASE}/avatar_checkpoints/texture/texture_gc_latest.pt",
    "Texture GE": f"{DRIVE_BASE}/avatar_checkpoints/texture/texture_ge_latest.pt",
    "LoRA VN": f"{DRIVE_BASE}/avatar_checkpoints/texture_lora/vietnamese_lora.pt",
}

print("🏭 Production Weights Summary:")
print("=" * 60)
total_size = 0
ready_count = 0

for name, fpath in production_files.items():
    p = Path(fpath)
    if p.exists():
        size_mb = p.stat().st_size / 1024 / 1024
        total_size += size_mb
        ready_count += 1
        print(f"  ✅ {name:15s} │ {size_mb:8.1f} MB │ {p.name}")
    else:
        print(f"  ⬜ {name:15s} │      --- │ Chưa train")

print("=" * 60)
print(f"  Tổng: {ready_count}/{len(production_files)} models │ {total_size:.1f} MB")

if ready_count == len(production_files):
    print("\n🎉 HOÀN TẤT! Tất cả weights đã sẵn sàng deploy.")
    print("   Bước tiếp: Tải các file .pt này về và bỏ vào thư mục weights/ của Backend API.")
else:
    print(f"\n⚠️ Còn thiếu {len(production_files) - ready_count} model(s). Tiếp tục training.")

In [ ]:
# === Cell 6.2: (Tùy chọn) Nén toàn bộ weights thành 1 file zip để tải về ===
import shutil
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/avatar_project"
export_dir = Path(f"{DRIVE_BASE}/avatar_export")
export_dir.mkdir(exist_ok=True)

# Copy tất cả weights vào 1 folder
sources = [
    f"{DRIVE_BASE}/avatar_checkpoints/geometry/geometry_vae_latest.pt",
    f"{DRIVE_BASE}/avatar_checkpoints/texture/texture_g_latest.pt",
    f"{DRIVE_BASE}/avatar_checkpoints/texture/texture_ga_latest.pt",
    f"{DRIVE_BASE}/avatar_checkpoints/texture/texture_gc_latest.pt",
    f"{DRIVE_BASE}/avatar_checkpoints/texture/texture_ge_latest.pt",
    f"{DRIVE_BASE}/avatar_checkpoints/texture_lora/vietnamese_lora.pt",
]

copied = 0
for src in sources:
    p = Path(src)
    if p.exists():
        shutil.copy2(str(p), str(export_dir / p.name))
        copied += 1

if copied > 0:
    zip_path = f"{DRIVE_BASE}/avatar_production_weights"
    shutil.make_archive(zip_path, 'zip', str(export_dir))
    zip_size = Path(f"{zip_path}.zip").stat().st_size / 1024 / 1024
    print(f"✅ Đã tạo: avatar_production_weights.zip ({zip_size:.1f} MB)")
    print(f"   Vị trí: {zip_path}.zip")
    print(f"   Chứa: {copied} model files")
    print(f"\n📥 Tải file zip này về máy rồi giải nén vào thư mục weights/ của Backend.")
else:
    print("❌ Không tìm thấy weights nào để xuất.")

---
## 🆘 Xử Lý Sự Cố

### Bị ngắt kết nối giữa chừng?
1. Mở lại notebook
2. Chạy **Phase 0** (Cell 0.1 + 0.2) để mount Drive lại
3. Chạy lại đúng Cell training bạn đang dở → AI tự resume

### Hết VRAM (Out of Memory)?
- Giảm `--batch-size` xuống `1`
- Giảm `--image-size` xuống `256`

### Training Loss không giảm?
- Thử giảm `--lr` (learning rate) xuống `5e-5`
- Kiểm tra dữ liệu: Chạy lại Cell 1.3 (validate)

### Muốn nâng cấp GPU?
- `Runtime → Change runtime type → A100` (cần Colab Pro)
- A100 nhanh gấp ~4-5 lần T4